# RAG with Unified Lineage — Working Demo (v0.0.5)

End-to-end: a single trace that spans data operations and LLM calls, with cross-domain links attributed automatically, plus a deterministic audit report at the end.

**v0.0.5** adds the audit JSON / Markdown exporter (`rudriq.export.audit`). The user just imports `rudriq.auto`, runs idiomatic pandas + openai code, and at the end calls `export_audit_json(run_id)` to get a schema-versioned report a compliance officer can read.

We use an httpx mock transport so the openai call runs offline.

## 1. Activate RudriQ

One import. Wires AutoLineage's `assign_id` callback into the linker, monkey-patches the OpenAI SDK, patches pandas getitem/tolist to propagate lineage IDs.

In [ ]:
import rudriq.auto

## 2. OTel setup

In production with Traceloop/OpenLLMetry installed, this is what `Traceloop.init()` would do. Without it, we set up a TracerProvider with the RudriQ SpanProcessor manually.

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from rudriq.processors import RudriQSpanProcessor

provider = TracerProvider()
rudriq_proc = RudriQSpanProcessor()
provider.add_span_processor(rudriq_proc)
trace.set_tracer_provider(provider)
tracer = trace.get_tracer('rag-demo')
print(f'RudriQ run_id: {rudriq_proc.run_id}')

## 3. Run the pipeline — pandas + openai, no manual instrumentation

AutoLineage's pandas hooks fire on `read_csv` and the filter, and through the new callback API auto-register each output's identity with the RudriQ linker. The lid-propagation patch carries the filter's lineage ID through `df['text'].tolist()`. The OpenAI SDK call captures its `input=` parameter into the span side-channel.

In [ ]:
import io, pandas as pd

csv = io.StringIO(
    'id,lang,text\n'
    '1,en,doc about machine learning\n'
    '2,fr,document sur lapprentissage\n'
    '3,en,doc about retrieval\n'
    '4,en,doc about graphs\n'
)
docs = pd.read_csv(csv)
docs = docs[docs['lang'] == 'en']
print(f'Filtered to {len(docs)} English docs.')

In [ ]:
import httpx, json
from openai import OpenAI

def fake_openai(request):
    n = len(json.loads(request.content)['input']) if request.content else 1
    return httpx.Response(200, json={
        'object': 'list', 'model': 'text-embedding-3-small',
        'data': [{'object': 'embedding', 'index': i, 'embedding': [0.1] * 8} for i in range(n)],
        'usage': {'prompt_tokens': 4 * n, 'total_tokens': 4 * n},
    })
client = OpenAI(api_key='sk-fake-offline-demo',
                http_client=httpx.Client(transport=httpx.MockTransport(fake_openai)))

with tracer.start_as_current_span('openai.embeddings.create') as span:
    span.set_attribute('gen_ai.system', 'openai')
    span.set_attribute('gen_ai.request.model', 'text-embedding-3-small')
    embeddings = client.embeddings.create(
        model='text-embedding-3-small',
        input=docs['text'].tolist(),
    )
print(f'Got {len(embeddings.data)} embeddings.')

## 4. Inspect the unified trace

The SpanProcessor persisted the LLM span. The lineage edge points from AutoLineage's lid back through the data pipeline.

**Scope note:** AutoLineage data nodes still live in AutoLineage's tracker (a separate store). v0.0.6 mirrors them into RudriQ's DuckDB so a single `load_run` returns the full graph. For now the audit chain shows them as `library=external`.

In [ ]:
from rudriq.storage import get_default_storage
import autolineage.auto as al_auto

graph = get_default_storage().load_run(rudriq_proc.run_id)
al_tracker = al_auto.get_tracker()

print(f'RudriQ DuckDB nodes (LLM side): {len(graph.nodes)}')
for n in graph.nodes:
    print(f'  {n.node_id[:20]:20s} | {n.kind.value:18s} | {n.library}.{n.operation}')

print(f'\nCross-domain edges: {len(graph.edges)}')
for e in graph.edges:
    parent_node = al_tracker.nodes.get(e.parent_id) if al_tracker else None
    parent_label = (
        f'AutoLineage[{parent_node["source"]}, shape={parent_node["shape"]}]'
        if parent_node else f'unknown({e.parent_id[:10]})'
    )
    print(f'  {parent_label}')
    print(f'    -> {e.child_id[:16]} | {e.kind.value} | {e.link_method.value} (conf={e.confidence:.2f})')

## 5. Generate the audit report (new in v0.0.5)

The `rudriq.export.audit` module renders the same trace as deterministic JSON or human-readable Markdown. JSON is for machine consumers (compliance tooling, diff between runs). Markdown is for an auditor reading the report directly. Same data, two views.

In [ ]:
from rudriq.export.audit import export_audit_json, export_audit_markdown

audit_json = export_audit_json(rudriq_proc.run_id)
audit_md = export_audit_markdown(rudriq_proc.run_id)

parsed = json.loads(audit_json)
print('=== Summary (what a compliance officer reads first) ===')
for k, v in parsed['summary'].items():
    print(f'  {k}: {v}')

print()
print('=== First 18 lines of Markdown report ===')
print('\n'.join(audit_md.splitlines()[:18]))

# Save to a file for inclusion in compliance documentation, hand-off
# to an auditor, or attachment to an incident postmortem.
with open('audit_report.md', 'w', encoding='utf-8') as f:
    f.write(audit_md)
print()
print(f'Full audit report ({len(audit_md)} chars) saved to audit_report.md')

## What just happened

RudriQ produced a cross-domain edge connecting the OpenAI call to the upstream DataFrame, with no manual `register_object_identity` and no manual `record_llm_input` in user code. Then the audit exporter rendered that trace as a schema-versioned JSON report and a human-readable Markdown report, both reproducible byte-for-byte across calls.

From the CLI: `rudriq audit --run-id <id> --format markdown --output report.md`.

v0.0.6 mirrors AutoLineage's data-side records into RudriQ's DuckDB so the audit chain shows full data→LLM ancestry without falling back to AutoLineage's tracker. v0.1 ships full Traceloop integration so the manual span context in cell 4 disappears.